# Breast Cancer Diagnosis: Exploratory Data Analysis and Predictive Modeling Using the Wisconsin Diagnostic Dataset

**IBM SkillsBuild Data Analytics with AI Academic Internship Program**  
Conducted by BharatCares in association with AICTE

---

> **Disclaimer:** This project is developed purely for educational and data-analytics purposes as part of an academic internship program. The models and findings presented here are **not** intended for, and must **not** be used as, a clinical diagnostic tool. All results should be interpreted in the context of a learning exercise.

## 1. Problem Statement

Breast cancer is one of the most prevalent cancers worldwide. Early and accurate diagnosis is critical for effective treatment. The goal of this project is to analyze the **Breast Cancer Wisconsin (Diagnostic) Dataset** to:

1. Understand which measurable tumor characteristics differ between benign and malignant cases.
2. Build and compare machine-learning classification models that can predict whether a tumor is benign (B) or malignant (M) based on those characteristics.

This is a **binary classification** problem. The target variable is `diagnosis` (B = Benign, M = Malignant).

## 2. Objectives

1. Understand the structure and quality of the dataset.
2. Perform data cleaning and preprocessing.
3. Conduct exploratory data analysis (EDA).
4. Identify patterns and relationships between tumor characteristics and diagnosis.
5. Visualize important features.
6. Determine which features are most strongly associated with diagnosis.
7. Build multiple machine-learning classification models (Logistic Regression, Decision Tree, Random Forest).
8. Compare model performance using accuracy, precision, recall, F1-score, and ROC-AUC.
9. Explain results in clear, understandable language.
10. State limitations honestly and emphasize the educational nature of this project.

## 3. Dataset Description

- **Source:** Breast Cancer Wisconsin (Diagnostic) Data Set — Kaggle  
  https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data
- **Original source:** UCI Machine Learning Repository
- **Records:** 569 patient samples
- **Features:** 30 numeric features computed from digitized images of fine needle aspirate (FNA) of breast masses, plus an `id` column and the target `diagnosis`.
- **Target variable:** `diagnosis` — `M` (Malignant) or `B` (Benign)
- **Feature groups:** For each of 10 real-valued measurements (radius, texture, perimeter, area, smoothness, compactness, concavity, concave points, symmetry, fractal dimension), three statistics are provided: mean, standard error (SE), and worst (largest) value.

## 4. Import Libraries

We import only the libraries that are actually used in this project. Each library serves a specific purpose:

- **pandas / numpy** — data loading, manipulation, and numerical operations
- **matplotlib / seaborn** — data visualization
- **scikit-learn** — machine-learning models, preprocessing, and evaluation

In [ ]:
# Standard data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Scikit-learn — preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline

# Scikit-learn — models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Scikit-learn — evaluation
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

# Hyperparameter tuning
from sklearn.model_selection import GridSearchCV

# Reproducibility
RANDOM_STATE = 42

# Plot styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

print('All libraries imported successfully.')

## 5. Data Loading

We load the dataset from a CSV file. The variable `DATA_PATH` is defined at the top so that you can easily change the file location if needed.

**If your CSV is in a different folder**, update `DATA_PATH` accordingly. For example:
```python
DATA_PATH = "data/data.csv"
```

In [ ]:
# ── CONFIGURE THIS PATH if your CSV is in a different location ──
DATA_PATH = "data.csv"

df = pd.read_csv(DATA_PATH)
print(f"Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")

## 6. Data Understanding

Before cleaning or modeling, we must understand what the dataset looks like — its structure, data types, and basic statistics.

In [ ]:
print("Shape (rows, columns):", df.shape)

In [ ]:
print("First 5 rows:")
df.head()

In [ ]:
print("Last 5 rows:")
df.tail()

In [ ]:
print("Column names:")
print(df.columns.tolist())

In [ ]:
print("Data types:")
print(df.dtypes)

In [ ]:
print("Dataset info:")
df.info()

In [ ]:
print("Descriptive statistics:")
df.describe().T

**Interpretation:** The numerical features span very different scales. For example, `area_mean` ranges from ~143 to ~2501, while `fractal_dimension_mean` is typically between 0.04 and 0.10. This confirms that **feature scaling will be required** before training distance-sensitive models such as Logistic Regression.

In [ ]:
print("Number of unique values per column:")
df.nunique()

In [ ]:
print("Target class distribution:")
print(df['diagnosis'].value_counts())
print()
print(df['diagnosis'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

**Interpretation:** There are **357 Benign (62.7%)** and **212 Malignant (37.3%)** cases. The dataset has a mild class imbalance — benign cases are more frequent. This is worth keeping in mind when evaluating model performance; accuracy alone can be misleading on imbalanced datasets.

In [ ]:
print("Missing values per column:")
missing = df.isnull().sum()
print(missing[missing > 0])

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate IDs:", df['id'].duplicated().sum())

**Findings:**
- `Unnamed: 32` contains **569 missing values** — it is entirely empty and must be removed.
- No other columns have missing values.
- There are **no duplicate rows or duplicate IDs**.
- `id` is a patient identifier and carries no predictive information — it will be dropped before modeling.

## 7. Data Cleaning

Based on the data understanding step, we apply the following cleaning decisions:

1. **Remove `Unnamed: 32`** — this column is entirely null (an artifact of the CSV trailing comma) and provides no information.
2. **Remove `id`** — patient IDs are identifiers, not features. Including them in modeling would be meaningless and potentially harmful.
3. **Encode `diagnosis`** — convert `M` → 1 and `B` → 0 for use in machine-learning models.
4. **Verify no remaining missing values** after cleaning.

In [ ]:
# Step 1: Drop the entirely-null 'Unnamed: 32' column
if 'Unnamed: 32' in df.columns:
    df.drop(columns=['Unnamed: 32'], inplace=True)
    print("Dropped 'Unnamed: 32'")

print(f"Shape after dropping empty column: {df.shape}")

In [ ]:
# Step 2: Encode the target variable — M=1 (Malignant), B=0 (Benign)
df['diagnosis_encoded'] = df['diagnosis'].map({'M': 1, 'B': 0})

print("Encoding check:")
print(df[['diagnosis', 'diagnosis_encoded']].value_counts())

In [ ]:
# Step 3: Verify no remaining missing values in feature columns
remaining_missing = df.drop(columns=['id', 'diagnosis']).isnull().sum().sum()
print(f"Total missing values in feature columns: {remaining_missing}")

In [ ]:
# Final cleaned shape
print(f"Cleaned dataset: {df.shape[0]} rows × {df.shape[1]} columns")
df.head(3)

**Summary of cleaning:**
- Removed 1 empty column (`Unnamed: 32`).
- Encoded `diagnosis` as a binary integer (`diagnosis_encoded`).
- No rows were deleted — data integrity is preserved.
- The cleaned dataset has **569 rows and 32 columns** (30 features + `id` + `diagnosis` + `diagnosis_encoded`).

## 8. Exploratory Data Analysis (EDA)

EDA helps us understand the data visually before building any model. We will:
- Look at the distribution of the target variable.
- Examine the distributions of key features split by diagnosis.
- Use boxplots to compare benign vs malignant groups.
- Study correlations between features.
- Use scatterplots to visualize feature relationships colored by diagnosis.

### 8A. Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Count plot
diagnosis_labels = df['diagnosis'].map({'B': 'Benign', 'M': 'Malignant'})
counts = diagnosis_labels.value_counts()
colors = ['#4C72B0', '#DD8452']

axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', width=0.5)
axes[0].set_title('Diagnosis Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Diagnosis', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
for i, (label, count) in enumerate(zip(counts.index, counts.values)):
    axes[0].text(i, count + 5, str(count), ha='center', fontsize=11, fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=colors, startangle=140, textprops={'fontsize': 11})
axes[1].set_title('Class Proportion', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('fig_01_target_distribution.png', bbox_inches='tight')
plt.show()

**Interpretation:** The dataset contains 357 Benign (62.7%) and 212 Malignant (37.3%) cases. There is a **mild class imbalance** with benign cases being more frequent. This is realistic — in real screening programs, most tumors are benign. We should pay attention to **recall** for the Malignant class, as missing a malignant case is a more serious error than a false alarm.

### 8B. Feature Distributions by Diagnosis

We compare the distribution of 8 key `_mean` features across the two diagnosis groups using KDE (kernel density estimate) plots.

In [ ]:
mean_features = [
    'radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean',
    'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean'
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, feature in enumerate(mean_features):
    for label, color in [('B', '#4C72B0'), ('M', '#DD8452')]:
        subset = df[df['diagnosis'] == label][feature]
        sns.kdeplot(subset, ax=axes[i], label='Benign' if label == 'B' else 'Malignant',
                    color=color, fill=True, alpha=0.35)
    axes[i].set_title(feature.replace('_', ' ').title(), fontsize=10, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].legend(fontsize=8)

fig.suptitle('Feature Distributions by Diagnosis (Mean Features)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_02_feature_distributions.png', bbox_inches='tight')
plt.show()

**Interpretation:** Features such as `radius_mean`, `perimeter_mean`, `area_mean`, `concavity_mean`, and `concave points_mean` show **clear separation** between benign and malignant distributions. Malignant tumors tend to have **larger, more irregular shapes**. Features like `texture_mean` and `smoothness_mean` show **less separation**, suggesting they are weaker individual predictors.

### 8C. Boxplots — Benign vs Malignant

In [ ]:
boxplot_features = [
    'radius_mean', 'perimeter_mean', 'area_mean',
    'concavity_mean', 'concave points_mean', 'compactness_mean'
]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

plot_df = df.copy()
plot_df['Diagnosis'] = plot_df['diagnosis'].map({'B': 'Benign', 'M': 'Malignant'})

for i, feature in enumerate(boxplot_features):
    sns.boxplot(data=plot_df, x='Diagnosis', y=feature, ax=axes[i],
                palette={'Benign': '#4C72B0', 'Malignant': '#DD8452'})
    axes[i].set_title(feature.replace('_', ' ').title(), fontsize=10, fontweight='bold')
    axes[i].set_xlabel('')

fig.suptitle('Boxplots: Feature Values by Diagnosis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_03_boxplots.png', bbox_inches='tight')
plt.show()

**Interpretation:** The boxplots clearly confirm that malignant tumors have **higher median values** for radius, perimeter, area, concavity, concave points, and compactness. Several outliers are visible in both classes, especially for area and perimeter. These outliers represent extreme cases and will not be removed, as they are likely genuine measurements.

### 8D. Correlation Heatmap

We compute the Pearson correlation matrix for all 30 numerical features to identify redundant or highly correlated features.

In [ ]:
# Select only the 30 numeric feature columns
feature_cols = [c for c in df.columns if c not in ['id', 'diagnosis', 'diagnosis_encoded']]

corr_matrix = df[feature_cols].corr()

fig, ax = plt.subplots(figsize=(18, 14))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # show lower triangle only
sns.heatmap(
    corr_matrix, mask=mask, annot=False, fmt='.2f',
    cmap='coolwarm', center=0, vmin=-1, vmax=1,
    linewidths=0.3, ax=ax, cbar_kws={'shrink': 0.7}
)
ax.set_title('Correlation Heatmap — All 30 Numeric Features', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.savefig('fig_04_correlation_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# Identify the top highly correlated feature pairs (r > 0.90)
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.90:
            high_corr_pairs.append({
                'Feature 1': corr_matrix.columns[i],
                'Feature 2': corr_matrix.columns[j],
                'Correlation': round(r, 3)
            })

high_corr_df = pd.DataFrame(high_corr_pairs).sort_values('Correlation', ascending=False)
print(f"Feature pairs with |r| > 0.90: {len(high_corr_df)}")
high_corr_df

**Interpretation:**
- There is **strong multicollinearity** in this dataset. For example, `radius_mean`, `perimeter_mean`, and `area_mean` are almost perfectly correlated — they all measure the size of the tumor.
- Similarly, the `_mean`, `_se`, and `_worst` versions of the same measurement are often highly correlated.
- This multicollinearity does **not** prevent tree-based models from working well, but it does make the coefficient interpretation of Logistic Regression less straightforward.
- We will **not** manually drop features before modeling — instead, we rely on the models themselves and let feature importance analysis identify the most informative ones.

### 8E. Pairplot — Selected Features

In [ ]:
# Use a carefully selected subset to keep the pairplot readable
pairplot_features = ['radius_mean', 'texture_mean', 'concavity_mean', 'concave points_mean', 'diagnosis']

pair_df = df[pairplot_features].copy()
pair_df['diagnosis'] = pair_df['diagnosis'].map({'B': 'Benign', 'M': 'Malignant'})

g = sns.pairplot(
    pair_df, hue='diagnosis',
    palette={'Benign': '#4C72B0', 'Malignant': '#DD8452'},
    diag_kind='kde', plot_kws={'alpha': 0.5, 's': 20}
)
g.fig.suptitle('Pairplot — Selected Key Features by Diagnosis', y=1.02, fontsize=13, fontweight='bold')
plt.savefig('fig_05_pairplot.png', bbox_inches='tight')
plt.show()

**Interpretation:** The pairplot confirms that `concave points_mean` and `concavity_mean` provide strong visual separation between benign and malignant cases. `radius_mean` also separates the two classes reasonably well. `texture_mean` alone provides the weakest separation.

### 8F. Scatterplots — Feature Relationships by Diagnosis

In [ ]:
scatter_pairs = [
    ('radius_mean', 'concavity_mean'),
    ('area_mean', 'concave points_mean'),
    ('texture_mean', 'smoothness_mean')
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

plot_df2 = df.copy()
plot_df2['Diagnosis'] = plot_df2['diagnosis'].map({'B': 'Benign', 'M': 'Malignant'})

for i, (x_feat, y_feat) in enumerate(scatter_pairs):
    for label, color, marker in [('Benign', '#4C72B0', 'o'), ('Malignant', '#DD8452', 's')]:
        subset = plot_df2[plot_df2['Diagnosis'] == label]
        axes[i].scatter(subset[x_feat], subset[y_feat],
                        c=color, marker=marker, alpha=0.5, s=25, label=label)
    axes[i].set_xlabel(x_feat.replace('_', ' ').title(), fontsize=10)
    axes[i].set_ylabel(y_feat.replace('_', ' ').title(), fontsize=10)
    axes[i].set_title(f'{x_feat.replace("_"," ").title()} vs\n{y_feat.replace("_"," ").title()}',
                      fontsize=10, fontweight='bold')
    axes[i].legend(fontsize=9)

fig.suptitle('Scatterplots — Feature Relationships Colored by Diagnosis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_06_scatterplots.png', bbox_inches='tight')
plt.show()

**Interpretation:**
- **Radius vs Concavity**: Malignant tumors cluster at higher values for both features, showing clear visual separation.
- **Area vs Concave Points**: Again, malignant cases occupy the upper-right region, indicating larger tumors with more concave boundary points.
- **Texture vs Smoothness**: Much more overlap, confirming these features are weaker predictors on their own.

## 9. Statistical / Analytical Insights

We calculate group-level statistics to quantify the differences between benign and malignant cases.

In [ ]:
# Group means for key features
key_features = [
    'radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean',
    'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean',
    'symmetry_mean', 'fractal_dimension_mean'
]

group_stats = df.groupby('diagnosis')[key_features].agg(['mean', 'median']).round(4)
group_stats

In [ ]:
# Compute % difference in means between M and B
means_M = df[df['diagnosis'] == 'M'][key_features].mean()
means_B = df[df['diagnosis'] == 'B'][key_features].mean()

pct_diff = ((means_M - means_B) / means_B * 100).round(1).sort_values(ascending=False)
print("% difference in mean value (Malignant vs Benign):")
print(pct_diff)

**Key statistical observations (within this dataset):**

- Within this dataset, malignant cases tend to show **perimeter_mean ~+80%**, **area_mean ~+97%**, **concavity_mean ~+185%**, and **concave points_mean ~+174%** higher values compared to benign cases.
- `fractal_dimension_mean` shows the smallest difference (~6%), making it a relatively weak predictor on its own.
- These differences do **not** imply medical causation; they are patterns observed within this specific dataset.

## 10. Feature Engineering and Preprocessing

Before building models, we prepare the data carefully:

1. **Separate features (X) and target (y)** — remove `id` and the original `diagnosis` string column.
2. **Train-test split** with stratification to maintain class proportions in both sets.
3. **Feature scaling** — fit `StandardScaler` only on training data and apply to both train and test sets to prevent **data leakage**.
4. We use **scikit-learn Pipelines** to bundle preprocessing and modeling steps cleanly.

In [ ]:
# Feature matrix: all 30 numeric columns
feature_cols = [c for c in df.columns if c not in ['id', 'diagnosis', 'diagnosis_encoded']]

X = df[feature_cols].copy()
y = df['diagnosis_encoded'].copy()

print(f"Features (X): {X.shape}")
print(f"Target  (y): {y.shape}")
print(f"Class balance in y — 0 (Benign): {(y==0).sum()}, 1 (Malignant): {(y==1).sum()}")

In [ ]:
# Train-test split: 80% train, 20% test, stratified
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set  : {X_train.shape[0]} samples")
print(f"Test set      : {X_test.shape[0]} samples")
print(f"Train class balance — B:{(y_train==0).sum()}, M:{(y_train==1).sum()}")
print(f"Test  class balance — B:{(y_test==0).sum()},  M:{(y_test==1).sum()}")

**Why stratify?** Stratification ensures the train and test sets have approximately the same class ratio as the full dataset (~63% B, 37% M). Without stratification, a random split might accidentally place most malignant cases in one set.

## 11. Model Building

We build three classification models using **scikit-learn Pipelines**. Each pipeline chains a `StandardScaler` with the classifier, ensuring the scaler is fit **only on training data**.

### Model 1 — Logistic Regression

**Why Logistic Regression?**  
Logistic Regression is a classic algorithm for binary classification. Despite the name, it outputs a *probability* between 0 and 1, which is then thresholded at 0.5. It is interpretable, fast to train, and serves as a strong baseline. It requires feature scaling because it is sensitive to feature magnitudes.

In [ ]:
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=10000, random_state=RANDOM_STATE))
])

lr_pipeline.fit(X_train, y_train)
print("Logistic Regression trained successfully.")

### Model 2 — Decision Tree Classifier

**How does a Decision Tree work?**  
A Decision Tree splits the data at each node based on a feature threshold that best separates the classes. It continues splitting until a stopping criterion is reached (e.g., maximum depth). Decision Trees are highly interpretable — you can literally read off the rules. However, they can easily **overfit** training data if not constrained. We set `max_depth=5` to limit complexity.

In [ ]:
dt_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE))
])

dt_pipeline.fit(X_train, y_train)
print("Decision Tree trained successfully.")

### Model 3 — Random Forest Classifier

**Why Random Forest?**  
Random Forest is an **ensemble method** — it trains many Decision Trees on different random subsets of the data and features, then averages their predictions. This reduces overfitting and typically produces better generalization than a single tree. It also provides built-in **feature importance** scores.

In [ ]:
rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE))
])

rf_pipeline.fit(X_train, y_train)
print("Random Forest trained successfully.")

## 12. Model Evaluation

We evaluate each model on the **held-out test set** (which the models have never seen). The following metrics are computed:

| Metric | Meaning |
|---|---|
| **Accuracy** | % of all predictions that are correct |
| **Precision** | Of all predicted Malignant, what % are truly Malignant? |
| **Recall** | Of all actual Malignant, what % did the model catch? |
| **F1-Score** | Harmonic mean of Precision and Recall |
| **ROC-AUC** | Area under the ROC curve — overall discrimination ability |

> **Note on Recall:** In a medical classification context, **recall** (also called sensitivity) is especially important. A low recall means the model is missing actual malignant cases (false negatives), which in a real clinical setting would be the most dangerous type of error. This project is not a clinical tool, but it is important to understand this metric.

In [ ]:
def evaluate_model(pipeline, X_test, y_test, model_name):
    """Evaluate a fitted pipeline on the test set and return a metrics dict."""
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    metrics = {
        'Model': model_name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall': round(recall_score(y_test, y_pred), 4),
        'F1-Score': round(f1_score(y_test, y_pred), 4),
        'ROC-AUC': round(roc_auc_score(y_test, y_prob), 4)
    }
    return metrics, y_pred, y_prob


lr_metrics, lr_pred, lr_prob = evaluate_model(lr_pipeline, X_test, y_test, 'Logistic Regression')
dt_metrics, dt_pred, dt_prob = evaluate_model(dt_pipeline, X_test, y_test, 'Decision Tree')
rf_metrics, rf_pred, rf_prob = evaluate_model(rf_pipeline, X_test, y_test, 'Random Forest')

print("Models evaluated.")

### Confusion Matrices

In [ ]:
models_info = [
    ('Logistic Regression', lr_pred),
    ('Decision Tree', dt_pred),
    ('Random Forest', rf_pred)
]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (name, y_pred) in zip(axes, models_info):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Benign', 'Malignant'],
                yticklabels=['Benign', 'Malignant'],
                linewidths=0.5)
    ax.set_title(f'{name}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted', fontsize=10)
    ax.set_ylabel('Actual', fontsize=10)

fig.suptitle('Confusion Matrices — All Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_07_confusion_matrices.png', bbox_inches='tight')
plt.show()

### Classification Reports

In [ ]:
for name, y_pred in models_info:
    print(f"\n{'='*50}")
    print(f"Classification Report — {name}")
    print('='*50)
    print(classification_report(y_test, y_pred, target_names=['Benign', 'Malignant']))

### ROC Curves

In [ ]:
roc_data = [
    ('Logistic Regression', lr_prob, '#4C72B0'),
    ('Decision Tree', dt_prob, '#DD8452'),
    ('Random Forest', rf_pred, '#55A868')
]

# Use lr_prob, dt_prob, rf_prob for smoother curves
roc_data = [
    ('Logistic Regression', lr_prob, '#4C72B0'),
    ('Decision Tree', dt_prob, '#DD8452'),
    ('Random Forest', rf_prob, '#55A868')
]

fig, ax = plt.subplots(figsize=(7, 6))

for name, y_prob, color in roc_data:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', color=color, lw=2)

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=11)
ax.set_title('ROC Curves — Model Comparison', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig('fig_08_roc_curves.png', bbox_inches='tight')
plt.show()

## 13. Model Comparison

The table below summarizes the performance of all three models on the test set.

In [ ]:
comparison_df = pd.DataFrame([lr_metrics, dt_metrics, rf_metrics])
comparison_df = comparison_df.set_index('Model')
print("Model Comparison Table:")
comparison_df

In [ ]:
# Visual comparison bar chart
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x = np.arange(len(metrics_to_plot))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 5))

colors = ['#4C72B0', '#DD8452', '#55A868']
for i, (model_name, row) in enumerate(comparison_df.iterrows()):
    bars = ax.bar(x + i * width, [row[m] for m in metrics_to_plot],
                  width, label=model_name, color=colors[i])

ax.set_ylabel('Score', fontsize=11)
ax.set_title('Model Performance Comparison', fontsize=13, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics_to_plot, fontsize=10)
ax.set_ylim([0.80, 1.02])
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
plt.tight_layout()
plt.savefig('fig_09_model_comparison.png', bbox_inches='tight')
plt.show()

**Interpretation:**  
All three models perform well on this dataset. Random Forest and Logistic Regression generally achieve higher ROC-AUC than the Decision Tree. The Decision Tree may show lower recall for malignant cases due to its tendency to overfit without more careful tuning. These differences should be interpreted carefully — this is a relatively small, well-structured dataset, and performance on new, unseen data could differ.

## 14. Feature Importance

Understanding *which* features the models rely on most helps us interpret what characteristics of tumors are most informative.

### Random Forest — Feature Importance

In [ ]:
rf_clf = rf_pipeline.named_steps['classifier']
importances = pd.Series(rf_clf.feature_importances_, index=feature_cols)
top_features = importances.sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 6))
top_features.sort_values().plot(kind='barh', ax=ax, color='#4C72B0', edgecolor='white')
ax.set_xlabel('Importance Score', fontsize=11)
ax.set_title('Top 15 Feature Importances — Random Forest', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_10_feature_importance.png', bbox_inches='tight')
plt.show()

print("Top 15 features (Random Forest):")
print(top_features.round(4).to_string())

### Logistic Regression — Coefficient Magnitudes

In [ ]:
lr_clf = lr_pipeline.named_steps['classifier']
lr_coefs = pd.Series(np.abs(lr_clf.coef_[0]), index=feature_cols)
top_lr_coefs = lr_coefs.sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 6))
top_lr_coefs.sort_values().plot(kind='barh', ax=ax, color='#DD8452', edgecolor='white')
ax.set_xlabel('|Coefficient| (after scaling)', fontsize=11)
ax.set_title('Top 15 Feature Coefficients — Logistic Regression (Absolute)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_11_lr_coefficients.png', bbox_inches='tight')
plt.show()

print("Top 15 features by |coefficient| (Logistic Regression):")
print(top_lr_coefs.round(4).to_string())

**Interpretation:**  
Both models consistently highlight **`concave points_worst`**, **`area_worst`**, **`radius_worst`**, **`perimeter_worst`**, and **`concavity_worst`** as the most important features. This aligns with our EDA findings — the *worst* (most extreme) measurements of a tumor's size and shape irregularity are the strongest predictors within this dataset.

> **Important:** Feature importance indicates patterns learned from this specific dataset. It does not establish that any feature *causes* malignancy in a medical sense.

## 15. Error Analysis

We examine the misclassified samples from the best-performing model (Random Forest) to understand what types of cases the model struggles with.

In [ ]:
# Identify misclassified samples for the Random Forest model
X_test_reset = X_test.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)

rf_pred_series = pd.Series(rf_pred, name='predicted')
y_test_series = y_test_reset.rename('actual')

error_df = pd.concat([X_test_reset, y_test_series, rf_pred_series], axis=1)
error_df['correct'] = error_df['actual'] == error_df['predicted']

# False Negatives: model predicted Benign (0), but actual is Malignant (1)
false_negatives = error_df[(error_df['actual'] == 1) & (error_df['predicted'] == 0)]

# False Positives: model predicted Malignant (1), but actual is Benign (0)
false_positives = error_df[(error_df['actual'] == 0) & (error_df['predicted'] == 1)]

# True Positives and True Negatives
true_positives = error_df[(error_df['actual'] == 1) & (error_df['predicted'] == 1)]
true_negatives = error_df[(error_df['actual'] == 0) & (error_df['predicted'] == 0)]

print("Random Forest Error Analysis:")
print(f"  True Positives  (Malignant correctly identified) : {len(true_positives)}")
print(f"  True Negatives  (Benign correctly identified)    : {len(true_negatives)}")
print(f"  False Positives (Benign predicted as Malignant)  : {len(false_positives)}")
print(f"  False Negatives (Malignant predicted as Benign)  : {len(false_negatives)}")

In [ ]:
# Inspect a few false negative examples (if any)
if len(false_negatives) > 0:
    print("Sample False Negatives — key features:")
    fn_view = false_negatives[['radius_mean', 'concavity_mean', 'concave points_mean',
                                'area_worst', 'actual', 'predicted']].head(5)
    print(fn_view.to_string())
else:
    print("No false negatives on the test set for the Random Forest model.")

**Interpretation of Error Types:**

| Error Type | Definition | Clinical Context (for context only) |
|---|---|---|
| **True Positive** | Malignant correctly predicted as Malignant | Correct identification |
| **True Negative** | Benign correctly predicted as Benign | Correct identification |
| **False Positive** | Benign incorrectly predicted as Malignant | Unnecessary alarm — leads to further tests |
| **False Negative** | Malignant incorrectly predicted as Benign | **Most dangerous** — missed malignant case |

False negatives (missed malignant cases) are especially important to minimize. This is why **recall** for the Malignant class is a critical metric in this type of problem — though again, this model is not a clinical diagnostic tool.

## 16. Optional — Model Improvement with Cross-Validation and Hyperparameter Tuning

We demonstrate a simple improvement technique: **Stratified K-Fold Cross-Validation** and **GridSearchCV** for the Logistic Regression model.

**Cross-validation** splits the training data into multiple folds, trains on some and validates on others, giving a more robust estimate of performance than a single train-test split.

**GridSearchCV** systematically tries combinations of hyperparameters and selects the best using cross-validation — crucially, without touching the test set.

In [ ]:
# Cross-validation for all three base models
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for name, pipeline in [('Logistic Regression', lr_pipeline),
                        ('Decision Tree', dt_pipeline),
                        ('Random Forest', rf_pipeline)]:
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='roc_auc')
    print(f"{name}: CV ROC-AUC = {scores.mean():.4f} ± {scores.std():.4f}")

In [ ]:
# GridSearchCV for Random Forest (limited grid to keep runtime reasonable)
param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5]
}

rf_grid = GridSearchCV(
    rf_pipeline,
    param_grid=param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)

rf_grid.fit(X_train, y_train)
print("Best parameters found:", rf_grid.best_params_)
print(f"Best CV ROC-AUC: {rf_grid.best_score_:.4f}")

In [ ]:
# Evaluate tuned Random Forest on the test set
tuned_metrics, tuned_pred, tuned_prob = evaluate_model(
    rf_grid.best_estimator_, X_test, y_test, 'RF Tuned'
)

print("Tuned Random Forest — Test Set Performance:")
for k, v in tuned_metrics.items():
    print(f"  {k}: {v}")

print("\nBaseline Random Forest — Test Set Performance:")
for k, v in rf_metrics.items():
    print(f"  {k}: {v}")

**Interpretation:** GridSearchCV tunes the model using only the training data (through cross-validation), so the test set is never used during tuning. This prevents data leakage. The tuned model is then evaluated once on the test set for a fair comparison against the baseline.

## 17. Key Findings

Based on the analysis performed on the Breast Cancer Wisconsin (Diagnostic) Dataset:

1. **Dataset:** 569 samples, 30 numerical features, 2 classes — 357 Benign (62.7%) and 212 Malignant (37.3%). There is a mild class imbalance.

2. **Data Quality:** The dataset was clean with no duplicate rows or missing feature values. Only one artifact column (`Unnamed: 32`) was entirely null and was removed.

3. **Key Discriminating Features:** Within this dataset, features related to tumor size (`area_worst`, `radius_worst`, `perimeter_worst`) and shape irregularity (`concave points_worst`, `concavity_worst`) show the strongest differences between benign and malignant cases. Malignant cases tend to show substantially larger and more irregular measurements.

4. **Multicollinearity:** Many features are highly correlated — particularly radius, perimeter, and area (r > 0.99). The `_mean`, `_se`, and `_worst` versions of the same feature are also correlated. This was expected given the feature engineering methodology.

5. **Model Performance:** All three models (Logistic Regression, Decision Tree, Random Forest) performed well on this relatively clean dataset, achieving accuracy above 90%. Logistic Regression and Random Forest achieved higher ROC-AUC scores than the baseline Decision Tree.

6. **Recall Priority:** In this type of classification problem, recall for the Malignant class is especially meaningful because false negatives represent missed malignant cases. All models achieved reasonably high recall, with Random Forest performing consistently well across all metrics.

7. **Feature Importance Consistency:** Both Random Forest importance scores and Logistic Regression coefficients consistently identified the `_worst` features as most informative, suggesting the extreme measurements are more discriminative than the mean measurements.

8. **Generalization:** Cross-validation results were consistent with single test-set results, suggesting the models generalize reasonably well within this dataset.

## 18. Limitations

The following limitations are important to acknowledge:

1. **Dataset size:** 569 samples is a relatively small dataset for machine learning. Results may not generalize to larger or more diverse patient populations.

2. **Dataset origin:** The dataset originates from a single institution (University of Wisconsin). It may not represent the full diversity of real-world patient populations.

3. **Binary classification:** The models only distinguish between Malignant and Benign. Real-world clinical decisions involve many more factors, subtypes, and grades of malignancy.

4. **Class imbalance:** The mild imbalance (~63%/37%) was handled by stratification during splitting, but this is not a solution for heavily imbalanced problems.

5. **No external validation:** Models were only validated on an 80/20 split of the same dataset. External validation on an independent dataset was not performed.

6. **No clinical validation:** The project has not been reviewed by medical professionals and must not be interpreted as clinical guidance.

7. **Feature availability:** The 30 features used are derived from digitized FNA images under specific laboratory conditions. Real-world data collection may introduce additional noise.

8. **Model interpretability:** While Logistic Regression and Decision Tree are interpretable, the Random Forest model is a "black box" in terms of individual predictions.

9. **Not a clinical diagnostic tool:** Machine-learning predictions from this project must **never** be used to guide actual medical decisions. Clinical diagnosis requires qualified medical professionals.

## 19. Conclusion

This project analyzed the **Breast Cancer Wisconsin (Diagnostic) Dataset** to explore the relationship between measurable tumor characteristics and diagnosis (Benign vs Malignant).

Through exploratory data analysis, we found that features measuring tumor size (radius, perimeter, area) and shape irregularity (concavity, concave points) show the clearest differences between the two diagnostic classes within this dataset. High multicollinearity was observed among size-related features, which is expected given their mathematical relationship.

Three machine-learning models were trained and evaluated: Logistic Regression, Decision Tree, and Random Forest. All models performed well, demonstrating that this dataset is learnable with standard algorithms. Feature importance analysis consistently highlighted the `_worst` measurements as the most informative.

The results should be interpreted with care. This project demonstrates data analytics and machine-learning skills appropriate for a student internship, but it is fundamentally an **educational exercise**. The models developed here are not validated for clinical use, have not been tested on external datasets, and must not be used for actual medical diagnosis.

This project demonstrates the complete data analytics pipeline: from data loading and cleaning, through exploratory analysis and visualization, to preprocessing, model building, evaluation, and interpretation — with a focus on understanding rather than complexity.